# Test 6.5: VarReg Comparison (Test B3)

**Гипотеза:** TargetVarianceRegularizer не влияет на variance при K=32 и может быть безопасно удалён.

**План:**
- Minimal test: K=32, N=5 runs, WITH vs WITHOUT VarReg
- Statistical comparison (Welch t-test)
- Cross-check с результатами activation ablation
- Optional: Full K-ablation без VarReg

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import json
from scipy import stats as sp_stats

print(f"TF version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

/Users/savenkovviktor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TF version: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
def logistic_map_step(x, r=3.99):
    return r * x * (1.0 - x)

def generate_logistic_dataset(num_images, image_size=28, r=3.99):
    dataset = []
    for _ in range(num_images):
        x = np.random.rand()
        seq = np.empty(image_size * image_size)
        for t in range(len(seq)):
            x = logistic_map_step(x, r)
            seq[t] = x
        dataset.append(seq.reshape((image_size, image_size)))
    return np.array(dataset, dtype='float32')[..., np.newaxis]

In [4]:
class KSparseLayer(layers.Layer):
    def __init__(self, k=32, **kwargs):
        super().__init__(**kwargs)
        self.k = k

    def call(self, inputs, training=None):
        latent_dim = tf.shape(inputs)[1]
        _, indices = tf.nn.top_k(tf.abs(inputs), k=self.k, sorted=False)
        mask = tf.reduce_sum(
            tf.one_hot(indices, latent_dim, dtype=inputs.dtype), axis=1
        )
        return inputs * mask

    def get_config(self):
        cfg = super().get_config()
        cfg['k'] = self.k
        return cfg


class TargetVarianceRegularizer(layers.Layer):
    def __init__(self, lambda_reg=0.01, target_variance=0.1, **kwargs):
        super().__init__(**kwargs)
        self.lambda_reg = lambda_reg
        self.target_variance = target_variance

    def call(self, inputs):
        var_per_dim = tf.math.reduce_variance(inputs, axis=0)
        mean_var = tf.reduce_mean(var_per_dim)
        self.add_loss(self.lambda_reg * tf.square(mean_var - self.target_variance))
        return inputs

    def get_config(self):
        cfg = super().get_config()
        cfg['lambda_reg'] = self.lambda_reg
        cfg['target_variance'] = self.target_variance
        return cfg


def chaos_activation(x):
    return tf.sin(8.0 * x) + 0.5 * tf.math.tanh(4.0 * x)

In [5]:
def build_ksparse_chaos_ae(latent_dim=128, k_active=32, use_varreg=False):
    input_img = keras.Input(shape=(28, 28, 1))
    x = layers.Flatten()(input_img)

    x = layers.Dense(256)(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.2)(x)

    latent_pre = layers.Dense(latent_dim)(x)
    latent_pre = layers.Activation(chaos_activation)(latent_pre)

    latent = KSparseLayer(k=k_active)(latent_pre)

    if use_varreg:
        latent = TargetVarianceRegularizer(
            lambda_reg=0.01,
            target_variance=0.1
        )(latent)

    encoder = keras.Model(input_img, latent)

    x = layers.Dense(256)(latent)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.1)(x)

    decoded = layers.Dense(28*28, activation='sigmoid')(x)
    decoded = layers.Reshape((28, 28, 1))(decoded)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder, encoder


def analyze_latent(encoder, images, zero_thresh=1e-6):
    latents = encoder.predict(images, verbose=0)
    var_per_dim = np.var(latents, axis=0)
    mean_var = float(np.mean(var_per_dim))
    dead = int(np.sum(np.all(np.abs(latents) < zero_thresh, axis=0)))
    sparsity = float(np.mean(np.abs(latents) < zero_thresh))

    return {
        'mean_variance': mean_var,
        'dead_neurons': dead,
        'sparsity': sparsity,
    }

In [6]:
np.random.seed(42)
train_data = generate_logistic_dataset(2000)
test_data = generate_logistic_dataset(500)
print(f"Train: {train_data.shape}, Test: {test_data.shape}")

Train: (2000, 28, 28, 1), Test: (500, 28, 28, 1)


In [7]:
n_runs = 5
results = {
    'with_varreg': [],
    'without_varreg': []
}

for condition_name, use_varreg in [
    ('with_varreg', True),
    ('without_varreg', False)
]:
    print(f"\n{condition_name.upper().replace('_', ' ')}:")
    print("-" * 60)

    for seed in range(n_runs):
        np.random.seed(seed)
        tf.random.set_seed(seed)

        ae, enc = build_ksparse_chaos_ae(
            latent_dim=128,
            k_active=32,
            use_varreg=use_varreg
        )

        ae.fit(
            train_data, train_data,
            epochs=10,
            batch_size=64,
            validation_split=0.1,
            verbose=0
        )

        s = analyze_latent(enc, test_data)
        results[condition_name].append(s)

        print(f"  Run {seed+1}/{n_runs}: var={s['mean_variance']:.4f}, "
              f"dead={s['dead_neurons']}")


WITH VARREG:
------------------------------------------------------------


2026-03-09 14:16:19.780424: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-03-09 14:16:19.780445: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2026-03-09 14:16:19.780449: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2026-03-09 14:16:19.780464: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-09 14:16:19.780474: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-03-09 14:16:20.169316: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  Run 1/5: var=0.4200, dead=0
  Run 2/5: var=0.4152, dead=0
  Run 3/5: var=0.4190, dead=0
  Run 4/5: var=0.4214, dead=0
  Run 5/5: var=0.4196, dead=0

WITHOUT VARREG:
------------------------------------------------------------
  Run 1/5: var=0.4179, dead=0
  Run 2/5: var=0.4192, dead=0
  Run 3/5: var=0.4203, dead=0
  Run 4/5: var=0.4169, dead=0
  Run 5/5: var=0.4185, dead=0


In [8]:
with_vars = [r['mean_variance'] for r in results['with_varreg']]
without_vars = [r['mean_variance'] for r in results['without_varreg']]

with_mean = np.mean(with_vars)
with_std = np.std(with_vars)
without_mean = np.mean(without_vars)
without_std = np.std(without_vars)

print(f"WITH VarReg:    {with_mean:.4f} +/- {with_std:.4f}")
print(f"WITHOUT VarReg: {without_mean:.4f} +/- {without_std:.4f}")
print(f"Difference:     {abs(with_mean - without_mean):.4f} "
      f"({abs(with_mean - without_mean)/with_mean*100:.1f}%)")

t_stat, p_value = sp_stats.ttest_ind(with_vars, without_vars)
print(f"\nt-test: t={t_stat:.3f}, p={p_value:.4f}")

if p_value > 0.05:
    print("NO SIGNIFICANT DIFFERENCE (p > 0.05)")
    print("  -> VarReg does not affect variance")
    print("  -> Safe to remove from architecture")
    decision = "SAFE_TO_REMOVE"
else:
    print("SIGNIFICANT DIFFERENCE (p < 0.05)")
    print("  -> VarReg has measurable effect")
    print("  -> Caution when removing")
    decision = "CAUTION"

# Cross-check with activation ablation
print(f"\n--- Cross-check with Activation Ablation ---")
print(f"Activation ablation (sin8+tanh4): 0.4183 +/- 0.0021")
print(f"This test (WITHOUT VarReg):       {without_mean:.4f} +/- {without_std:.4f}")

diff_from_ablation = abs(without_mean - 0.4183)
if diff_from_ablation < 0.01:
    print(f"CONSISTENT (diff = {diff_from_ablation:.4f})")
    consistency = "CONSISTENT"
else:
    print(f"DIVERGENT (diff = {diff_from_ablation:.4f})")
    consistency = "DIVERGENT"

analysis = {
    'with_mean': with_mean,
    'with_std': with_std,
    'without_mean': without_mean,
    'without_std': without_std,
    'p_value': p_value,
    'decision': decision,
    'consistency': consistency
}

WITH VarReg:    0.4190 +/- 0.0021
WITHOUT VarReg: 0.4186 +/- 0.0011
Difference:     0.0005 (0.1%)

t-test: t=0.404, p=0.6966
NO SIGNIFICANT DIFFERENCE (p > 0.05)
  -> VarReg does not affect variance
  -> Safe to remove from architecture

--- Cross-check with Activation Ablation ---
Activation ablation (sin8+tanh4): 0.4183 +/- 0.0021
This test (WITHOUT VarReg):       0.4186 +/- 0.0011
CONSISTENT (diff = 0.0003)


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bp = axes[0].boxplot(
    [with_vars, without_vars],
    tick_labels=['WITH\nVarReg', 'WITHOUT\nVarReg'],
    widths=0.6,
    patch_artist=True
)

for patch, color in zip(bp['boxes'], ['lightcoral', 'lightgreen']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_ylabel('Mean Variance', fontsize=12, fontweight='bold')
axes[0].set_title(f'K=32: WITH vs WITHOUT VarReg (N={n_runs} runs)',
                  fontweight='bold', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')

if p_value > 0.05:
    sig_text = f"p={p_value:.3f}\n(NOT significant)"
    color = 'green'
else:
    sig_text = f"p={p_value:.3f}\n(significant)"
    color = 'red'

axes[0].text(0.5, 1.02, sig_text, ha='center', va='bottom', fontsize=10,
            transform=axes[0].transAxes,
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))

# Individual runs
x_with = np.ones(len(with_vars)) * 1
x_without = np.ones(len(without_vars)) * 2

axes[1].scatter(x_with, with_vars, s=80, alpha=0.7,
               color='lightcoral', edgecolors='black', linewidths=1,
               label='WITH VarReg')
axes[1].scatter(x_without, without_vars, s=80, alpha=0.7,
               color='lightgreen', edgecolors='black', linewidths=1,
               label='WITHOUT VarReg')

for i in range(len(with_vars)):
    axes[1].plot([1, 2], [with_vars[i], without_vars[i]],
                'k--', alpha=0.3, linewidth=0.5)

axes[1].plot([0.8, 1.2], [with_mean]*2, 'r-', linewidth=3,
            label=f"Mean WITH: {with_mean:.3f}")
axes[1].plot([1.8, 2.2], [without_mean]*2, 'g-', linewidth=3,
            label=f"Mean WITHOUT: {without_mean:.3f}")

axes[1].set_xlim(0.5, 2.5)
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['WITH\nVarReg', 'WITHOUT\nVarReg'])
axes[1].set_ylabel('Mean Variance', fontsize=12, fontweight='bold')
axes[1].set_title('Individual Runs Comparison', fontweight='bold', fontsize=13)
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Test B3: TargetVarianceRegularizer Effect on K=32',
            fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('test_b3_varreg_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("Saved: test_b3_varreg_comparison.png")

Saved: test_b3_varreg_comparison.png


In [10]:
output = {
    'minimal_test': {
        'with_varreg': [r['mean_variance'] for r in results['with_varreg']],
        'without_varreg': [r['mean_variance'] for r in results['without_varreg']],
        'analysis': {
            'with_mean': analysis['with_mean'],
            'without_mean': analysis['without_mean'],
            'p_value': analysis['p_value'],
            'decision': analysis['decision'],
            'consistency': analysis['consistency']
        }
    }
}

with open('test_b3_results.json', 'w') as f:
    json.dump(output, f, indent=2)